# Dry Bean Type Classification using Machine Learning

**Objective:** An agriculture company classifies dry beans manually, which is slow, costly and error-prone.
This notebook builds a supervised multiclass classifier that predicts the bean type
(*Seker, Barbunya, Bombay, Cali, Dermason, Horoz, Sira*) from 16 camera-derived shape measurements,
so the process can be automated and run in real time.

**Notebook outline**
1. Import and load the data
2. Exploratory data analysis (EDA)
3. Missing values and outlier analysis
4. Feature engineering and preprocessing
5. Model building (multiple classifiers + cross-validation)
6. Handling class imbalance
7. Model evaluation and overfitting check
8. Hyperparameter tuning
9. Model comparison table
10. Streamlit classifier app

# 1. Import and Load the Data

### Importing Libraries

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import loguniform, randint

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set(style="darkgrid", font_scale=1.2)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

Core libraries for data handling (`numpy`, `pandas`), statistics (`scipy`), visualisation (`matplotlib`, `seaborn`)
and model persistence (`joblib`). A global `RANDOM_STATE` is defined so every split, model and search is reproducible.

### Machine learning libraries and performance metrics

In [ ]:
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     RandomizedSearchCV)
from sklearn.preprocessing import LabelEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
                              BaggingClassifier, HistGradientBoostingClassifier, VotingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

# Imbalance handling (SMOTE / random over- and under-sampling) needs imbalanced-learn:
#   pip install imbalanced-learn
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not installed -> only class-weight balancing will be run in Section 6.")

All scikit-learn components used later: model-selection tools, preprocessing, the seven required classifier families
(logistic regression, tree, forest, KNN, SVM, ensembles, Naive Bayes) plus a few extras, and the evaluation metrics.
`imbalanced-learn` is imported inside a `try/except` so the notebook still runs without it; install it to enable SMOTE and random over/under-sampling.

### Library versions

In [ ]:
import sklearn, matplotlib
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scipy      :", __import__("scipy").__version__)
print("seaborn    :", sns.__version__)
print("matplotlib :", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
if HAS_IMBLEARN:
    import imblearn; print("imblearn   :", imblearn.__version__)

Recording library versions makes the experiment reproducible and helps debug environment differences.

### Loading the dataset

In [ ]:
DATA_PATH = "Dry_Bean_Dataset.xlsx"          # keep the file next to this notebook
df = pd.read_excel(DATA_PATH, sheet_name="Dry_Beans_Dataset")

# Tidy two column names / labels from the raw file (typo + inconsistent capitalisation)
df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})
df["Class"] = df["Class"].str.title()

The Excel sheet is loaded with `pd.read_excel`. The raw column `AspectRation` (typo) is renamed to `AspectRatio`,
`roundness` to `Roundness`, and class labels are converted from `SEKER` to `Seker` style for nicer plots.

In [ ]:
df.head()

`head()` shows the first rows: 16 numeric shape features and the `Class` target.

In [ ]:
df.tail()

`tail()` confirms the end of the file is as well-formed as the start (no footer rows or stray values).

In [ ]:
df.sample(5, random_state=RANDOM_STATE)

A random sample guards against the first/last rows being unrepresentative (the file is sorted by class).

### Dimensions, data types and descriptive statistics

In [ ]:
print('Rows x Columns:', df.shape)

In [ ]:
df.info()

`info()` shows the dimensions, dtypes and non-null counts. All 16 predictors are numeric (`int64`/`float64`),
`Class` is the only categorical column, and no column has nulls.

In [ ]:
features = [c for c in df.columns if c != "Class"]
print(len(features), "features:", features)
df[features].describe().T

`describe()` reveals very different scales: `Area` and `ConvexArea` are in the tens of thousands of pixels, while
`Solidity`, `Extent`, `Eccentricity` and the ShapeFactors lie between 0 and 1. Distance-based models (KNN, SVM) and
linear models therefore **need scaling**. Several features also show max values far above the 75th percentile (right skew / outliers).

# 2. Exploratory Data Analysis (EDA)

### Class distribution (checking for imbalance)

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = (class_counts / len(df) * 100).round(2)
display(pd.DataFrame({"Count": class_counts, "Percent": class_pct}))
print(f"Imbalance ratio (largest / smallest class): {class_counts.max() / class_counts.min():.1f} : 1")

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.countplot(data=df, x="Class", order=class_counts.index, palette="viridis", ax=ax[0])
ax[0].set_title("Number of beans per class")
for p in ax[0].patches:
    ax[0].annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                   ha="center", va="bottom", fontsize=10)
ax[1].pie(class_counts, labels=class_counts.index, autopct="%1.1f%%", startangle=90,
          colors=sns.color_palette("viridis", len(class_counts)))
ax[1].set_title("Class share")
plt.tight_layout(); plt.show()

The target is **imbalanced**: Dermason is the largest class while Bombay is by far the smallest.
Accuracy alone can hide poor performance on small classes, so we will track **macro-F1** and per-class recall,
and test imbalance-handling techniques in Section 6.

### Distribution of each feature (histograms)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(df[col], kde=True, bins=40, color="teal", ax=ax)
    ax.set_title(f"{col} (skew = {df[col].skew():.2f})", fontsize=12)
    ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

Histograms show that many size features (`Area`, `ConvexArea`, `EquivDiameter`, `MinorAxisLength`) are right-skewed,
while `Solidity`, `ShapeFactor4`, `Extent` and `Eccentricity` are left-skewed. This informs the skewness treatment in Section 4.

### Outlier view: boxplots of each feature

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(y=df[col], color="skyblue", ax=ax)
    ax.set_title(col, fontsize=12); ax.set_ylabel("")
plt.tight_layout(); plt.show()

Boxplots show many points beyond the whiskers in almost every feature; these are quantified and treated in Section 3.

### Feature behaviour per class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, col in zip(axes.ravel(), ["Area", "Perimeter", "MajorAxisLength", "AspectRatio", "Solidity", "ShapeFactor1"]):
    sns.boxplot(data=df, x="Class", y=col, order=sorted(df["Class"].unique()), palette="Set2", ax=ax)
    ax.set_title(f"{col} by class"); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

Size features (`Area`, `Perimeter`, `MajorAxisLength`) separate **Bombay** (very large) and **Dermason** (small) well,
but overlap heavily among the mid-sized classes (Barbunya / Cali / Horoz / Sira / Seker). Shape features such as `AspectRatio` and `ShapeFactor1` help to separate those overlapping classes.

### Correlation heatmap

In [ ]:
corr = df[features].corr()
plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 9})
plt.title("Feature correlation matrix"); plt.show()

pairs = (corr.where(~mask).stack().rename("corr").reset_index()
         .rename(columns={"level_0": "Feature 1", "level_1": "Feature 2"}))
display(pairs[pairs["corr"].abs() > 0.95].sort_values("corr", key=abs, ascending=False).reset_index(drop=True))

Many features are almost perfectly correlated (e.g. `Area` & `ConvexArea`, `Compactness` & `ShapeFactor3`, `Perimeter` & `EquivDiameter`)
because they are geometric functions of each other. This **multicollinearity** can destabilise linear models but does not hurt tree-based models,
and we keep all features so the Streamlit app can ask for the same measurements the camera system produces.

### Multivariate relationships (pairplot)

In [ ]:
pair_cols = ["Area", "MajorAxisLength", "AspectRatio", "Solidity", "ShapeFactor1", "Class"]
sample = df[pair_cols].sample(1500, random_state=RANDOM_STATE)       # sample to keep the plot fast
sns.pairplot(sample, hue="Class", corner=True, plot_kws={"alpha": 0.5, "s": 15}, palette="tab10")
plt.show()

A pairplot of six representative features (on a 1,500-row sample for speed) shows that classes form partly separable clusters in
multi-dimensional space, with visible overlap between some classes (e.g. Dermason vs Sira, Barbunya vs Cali, Seker vs Horoz). Non-linear models may therefore help, although a good linear boundary can already capture a lot of this structure.

### Summary of key EDA findings
- **No missing values**, 16 numeric predictors, one categorical target with 7 classes.
- **Class imbalance:** Dermason is the majority class and Bombay the minority class.
- **Different scales** (pixels vs ratios) -> feature scaling is required.
- **Skewed distributions and many IQR outliers** in almost every feature.
- **Strong multicollinearity** among geometric features (area/perimeter family, aspect ratio/compactness/shape factor 3).
- **Bombay** is easy to identify by size; the harder task is separating look-alike beans such as **Dermason vs Sira** and **Barbunya vs Cali**.

# 3. Missing Values and Outlier Treatment

### Missing values and duplicates

In [ ]:
print("Total missing values:", int(df.isnull().sum().sum()))
dups = df.duplicated().sum()
print("Duplicate rows      :", dups)
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

There are **no missing values**, so no imputation is needed. Exact duplicate rows are removed because identical records can appear in both the
train and test split and inflate test scores.

### Outlier detection: IQR and Z-score

In [ ]:
X_all = df[features]
q1, q3 = X_all.quantile(0.25), X_all.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_mask = (X_all < lower) | (X_all > upper)
z_mask = pd.DataFrame(np.abs(stats.zscore(X_all)) > 3, columns=features)

outlier_summary = pd.DataFrame({
    "IQR outliers": iqr_mask.sum(),
    "IQR %": (iqr_mask.mean() * 100).round(2),
    "Z-score (|z|>3) outliers": z_mask.sum(),
})
display(outlier_summary)
print(f"Rows with at least one IQR outlier: {iqr_mask.any(axis=1).mean() * 100:.1f}%")
print(f"Rows with at least one |z|>3 value: {z_mask.any(axis=1).mean() * 100:.1f}%")

Both methods flag outliers in nearly every column. Because a sizeable share of rows has at least one flagged value, **deleting them would throw away a lot of data**.

### Which classes do the outliers belong to?

In [ ]:
flagged = pd.DataFrame({"Class": df["Class"], "Has IQR outlier": iqr_mask.any(axis=1)})
display((flagged.groupby("Class")["Has IQR outlier"].mean() * 100).round(1).rename("% rows with outlier").to_frame())

Outliers are not spread evenly: some classes have a much higher share of flagged rows. Many of these are genuine natural variation of that bean type
(e.g. unusually large beans) rather than measurement errors. Deleting those rows would hurt exactly the minority classes we care about.

**Decision:** keep all rows and **cap (winsorise) values at the IQR fences** (Q1 - 1.5 IQR, Q3 + 1.5 IQR). Capping limits the influence of extreme values on scale-sensitive models while preserving every sample.
To avoid data leakage the fences are learned **on the training set only**, inside the modelling pipeline (Section 4).

Note: every Bombay bean is flagged by the IQR rule simply because Bombay beans are much larger than the rest, a good reason **not** to drop outlier rows. In a quick check with an SVM, capping vs not capping changed cross-validated macro-F1 by well under half a percentage point, so it is a safe, low-impact step rather than a key driver of accuracy.

# 4. Feature Engineering and Preprocessing

### Encoding the target

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df["Class"])
X = df[features].copy()
class_names = list(le.classes_)
print(dict(zip(class_names, le.transform(class_names))))

The only categorical column is the target `Class`; it is label-encoded to integers 0-6. Predictors need no encoding.

### Stratified train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, " Test:", X_test.shape)
pd.DataFrame({
    "Train %": pd.Series(y_train).map(dict(enumerate(class_names))).value_counts(normalize=True).mul(100).round(2),
    "Test %":  pd.Series(y_test).map(dict(enumerate(class_names))).value_counts(normalize=True).mul(100).round(2),
})

An 80/20 split with `stratify=y` keeps the class proportions identical in train and test, which matters for a small class like Bombay.
**The split is done before any fitting of scalers or cappers**, so no information from the test set leaks into preprocessing.

### Custom outlier-capping transformer

In [ ]:
# The capper is saved in its own module so the Streamlit app can un-pickle the trained pipeline.
Path("bean_utils.py").write_text("""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class IQRCapper(BaseEstimator, TransformerMixin):
    \"\"\"Clip every column to [Q1 - k*IQR, Q3 + k*IQR] learned from the training data.\"\"\"

    def __init__(self, k=1.5):
        self.k = k

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
        q1, q3 = X.quantile(0.25), X.quantile(0.75)
        iqr = q3 - q1
        self.lower_ = (q1 - self.k * iqr).values
        self.upper_ = (q3 + self.k * iqr).values
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        return pd.DataFrame(np.clip(X.values, self.lower_, self.upper_),
                            columns=self.columns_, index=X.index)
""")

from bean_utils import IQRCapper

`IQRCapper` is a scikit-learn transformer: `fit` learns the IQR fences from the training data, `transform` clips any dataframe to them.
Writing it to `bean_utils.py` means the same class can be imported by the Streamlit app later.

### Checking skewness

In [ ]:
capper = IQRCapper().fit(X_train)
X_train_capped = capper.transform(X_train)

skew_table = pd.DataFrame({"Skew (raw)": X_train.skew(), "Skew (after capping)": X_train_capped.skew()}).round(2)
skew_table["Treat?"] = np.where(skew_table["Skew (after capping)"].abs() > 0.5, "Yes", "No")
display(skew_table)

skewed_cols = skew_table.index[skew_table["Treat?"] == "Yes"].tolist()
other_cols = [c for c in features if c not in skewed_cols]
print("Skewed columns to transform:", skewed_cols)

Skewness is measured on the capped training data. Features with |skew| > 0.5 (a common rule of thumb for moderate skew) are marked for a
**Yeo-Johnson power transform**, which makes the distribution closer to normal and also standardises it. The remaining, roughly symmetric features are simply standardised with `StandardScaler`.

### Preprocessing pipeline: cap outliers, fix skew, scale

In [ ]:
def make_preprocessor():
    return Pipeline([
        ("cap", IQRCapper()),
        ("scale", ColumnTransformer([
            ("power", PowerTransformer(method="yeo-johnson", standardize=True), skewed_cols),
            ("std", StandardScaler(), other_cols),
        ])),
    ])

# Inspect the effect on the training data
prep_demo = make_preprocessor().fit(X_train)
X_train_proc = pd.DataFrame(prep_demo.transform(X_train), columns=skewed_cols + other_cols)
display(pd.DataFrame({"Skew before": X_train[skewed_cols + other_cols].skew(),
                      "Skew after": X_train_proc.skew()}).round(2))
display(X_train_proc.describe().loc[["mean", "std"]].round(2))

After preprocessing, skewness drops toward zero and every feature has mean ~0 and std ~1.
The preprocessor is wrapped in a function so that **each model gets its own fresh pipeline** (`preprocess -> classifier`).
Putting preprocessing inside the pipeline is what makes cross-validation honest: capping and scaling are re-fitted on each training fold only.

# 5. Model Building: Try Multiple Classifiers

### Defining the models and evaluation helper

In [ ]:
def make_pipeline(clf, sampler=None):
    """preprocessing -> (optional sampler) -> classifier."""
    if sampler is None:
        return Pipeline([("prep", make_preprocessor()), ("clf", clf)])
    return ImbPipeline([("prep", make_preprocessor()), ("sampler", sampler), ("clf", clf)])


def get_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "Decision Tree":       DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest":       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
        "KNN":                 KNeighborsClassifier(n_neighbors=7),
        "SVM":                 SVC(kernel="rbf", random_state=RANDOM_STATE),
        "Naive Bayes":         GaussianNB(),
        "LDA":                 LinearDiscriminantAnalysis(),
        "Extra Trees":         ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
        "AdaBoost":            AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "Bagging (Trees)":     BaggingClassifier(DecisionTreeClassifier(), n_estimators=50, n_jobs=-1, random_state=RANDOM_STATE),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "Voting (SVM+RF+HGB)": VotingClassifier(
            estimators=[("svm", SVC(random_state=RANDOM_STATE)),
                        ("rf", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)),
                        ("hgb", HistGradientBoostingClassifier(random_state=RANDOM_STATE))],
            voting="hard"),
    }


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
OVERFIT_GAP = 0.03      # train - test accuracy gap above this is flagged as overfitting


def evaluate(name, pipe, cv=True):
    """Fit on train, score on train/test, and (optionally) 5-fold stratified CV on the training data."""
    t0 = time.time()
    pipe.fit(X_train, y_train)
    pred_tr, pred_te = pipe.predict(X_train), pipe.predict(X_test)
    row = {
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, pred_tr),
        "Test Accuracy": accuracy_score(y_test, pred_te),
        "Precision (macro)": precision_score(y_test, pred_te, average="macro"),
        "Recall (macro)": recall_score(y_test, pred_te, average="macro"),
        "F1 (macro)": f1_score(y_test, pred_te, average="macro"),
        "F1 (weighted)": f1_score(y_test, pred_te, average="weighted"),
    }
    if cv:
        scores = cross_val_score(clone(pipe), X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1)
        row["CV Accuracy (mean)"], row["CV Accuracy (std)"] = scores.mean(), scores.std()
    row["Gap (train-test)"] = row["Train Accuracy"] - row["Test Accuracy"]
    row["Overfitting (Y/N)"] = "Y" if row["Gap (train-test)"] > OVERFIT_GAP else "N"
    row["Time (s)"] = time.time() - t0
    return row, pipe

- **`make_pipeline`** chains preprocessing and a classifier (optionally a sampler, used in Section 6).
- **`get_models`** returns the classifiers: logistic regression, decision tree, random forest, KNN, SVM, Naive Bayes, LDA, plus ensembles
  (extra trees, AdaBoost, bagging, histogram gradient boosting, hard-voting ensemble).
- **`evaluate`** fits a model and records train / test accuracy, macro precision / recall / F1, weighted F1, and **5-fold stratified cross-validation accuracy** on the training set.
  A model is flagged as overfitting when train accuracy exceeds test accuracy by more than 3 percentage points.

### Training and evaluating all models

In [ ]:
results, fitted = [], {}
for name, clf in get_models().items():
    row, pipe = evaluate(name, make_pipeline(clf))
    results.append(row); fitted[name] = pipe
    print(f"{name:<24} test acc = {row['Test Accuracy']:.4f} | CV acc = {row['CV Accuracy (mean)']:.4f} | macro-F1 = {row['F1 (macro)']:.4f}")

results_df = pd.DataFrame(results).set_index("Model").sort_values("F1 (macro)", ascending=False)
results_df.round(4)

Every model is trained on the preprocessed training data and scored on the untouched test set. Cross-validation accuracy (mean and std over 5 folds)
is reported next to the test accuracy: when the two agree, the test score is not a lucky split. Results are sorted by macro-F1.

### Cross-validation vs test accuracy

In [ ]:
plot_df = results_df.reset_index().melt(id_vars="Model", value_vars=["CV Accuracy (mean)", "Test Accuracy"],
                                        var_name="Metric", value_name="Accuracy")
plt.figure(figsize=(14, 6))
sns.barplot(data=plot_df, y="Model", x="Accuracy", hue="Metric", palette="Set2")
plt.xlim(plot_df["Accuracy"].min() - 0.03, 1.0)
plt.title("Cross-validation accuracy vs test accuracy"); plt.show()

Models whose CV accuracy and test accuracy are close generalise consistently. The plot also makes it easy to see which families
(kernel / ensemble models) lead and which (Naive Bayes, AdaBoost, LDA) lag.

# 6. Handling Class Imbalance

We compare strategies on **two strong models (SVM and Random Forest)**:
1. **Baseline** (no handling)
2. **Class weighting** (`class_weight="balanced"`)
3. **Random over-sampling** (duplicate minority samples)
4. **Random under-sampling** (drop majority samples)
5. **SMOTE** (synthesise new minority samples)

Samplers sit *inside* the pipeline after preprocessing, so they are only applied to the training part of each fold and never to validation or test data.

In [ ]:
minority_class = class_counts.idxmin()
print("Minority class:", minority_class)

def make_model(kind, weighted=False):
    cw = "balanced" if weighted else None
    if kind == "SVM":
        return SVC(kernel="rbf", class_weight=cw, random_state=RANDOM_STATE)
    return RandomForestClassifier(n_estimators=200, class_weight=cw, n_jobs=-1, random_state=RANDOM_STATE)

strategies = {"Baseline": dict(), "Class weight": dict(weighted=True)}
if HAS_IMBLEARN:
    strategies.update({
        "Random Over-sampling":  dict(sampler=RandomOverSampler(random_state=RANDOM_STATE)),
        "Random Under-sampling": dict(sampler=RandomUnderSampler(random_state=RANDOM_STATE)),
        "SMOTE":                 dict(sampler=SMOTE(random_state=RANDOM_STATE)),
    })

rows = []
for kind in ["SVM", "Random Forest"]:
    for strat, kw in strategies.items():
        pipe = make_pipeline(make_model(kind, kw.get("weighted", False)), kw.get("sampler"))
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rep = classification_report(y_test, pred, target_names=class_names, output_dict=True)
        cv_f1 = cross_val_score(clone(pipe), X_train, y_train, cv=skf, scoring="f1_macro", n_jobs=-1).mean()
        rows.append({"Model": kind, "Strategy": strat,
                     "Test Accuracy": accuracy_score(y_test, pred),
                     "F1 (macro)": rep["macro avg"]["f1-score"],
                     "CV F1 (macro)": cv_f1,
                     f"{minority_class} Recall": rep[minority_class]["recall"],
                     f"{minority_class} F1": rep[minority_class]["f1-score"],
                     "Worst-class F1": min(rep[c]["f1-score"] for c in class_names)})

imbalance_df = pd.DataFrame(rows)
imbalance_df.round(4)

For each strategy we compare accuracy, **macro-F1** (treats all classes equally), cross-validated macro-F1, recall/F1 for the minority class,
and the F1 of the worst-performing class. If a strategy raises minority-class recall without lowering macro-F1 it is genuinely helping.

In [ ]:
plt.figure(figsize=(14, 5))
sns.barplot(data=imbalance_df, x="Strategy", y="F1 (macro)", hue="Model", palette="Set1")
plt.ylim(imbalance_df["F1 (macro)"].min() - 0.02, imbalance_df["F1 (macro)"].max() + 0.01)
plt.xticks(rotation=15); plt.title("Macro-F1 by imbalance strategy"); plt.show()

The chart summarises whether any resampling/weighting strategy beats the baseline.

**Reading the results (reference run, class weighting only):** class weighting moved macro-F1 by less than 0.2 percentage points for both models (up slightly for SVM, down slightly for Random Forest), and the minority class **Bombay was already classified perfectly (recall = F1 = 1.0) at baseline**. The imbalance (about 6.8 : 1 between the largest and smallest class) is mild and the minority class is easy to separate, so none of these techniques is essential here. The remaining weak spot is the worst-class F1 (Sira), which is a class-overlap problem rather than a class-size problem. If you have `imbalanced-learn` installed, compare the extra Over-sampling / Under-sampling / SMOTE rows against the baseline in the same way.

# 7. Model Evaluation and Overfitting Check

### Train vs test accuracy for every model

In [ ]:
gap_df = results_df.reset_index().melt(id_vars="Model", value_vars=["Train Accuracy", "Test Accuracy"],
                                       var_name="Set", value_name="Accuracy")
plt.figure(figsize=(14, 6))
sns.barplot(data=gap_df, y="Model", x="Accuracy", hue="Set", palette="coolwarm")
plt.xlim(gap_df["Accuracy"].min() - 0.03, 1.005)
plt.title("Train vs test accuracy (large gap = overfitting)"); plt.show()

results_df[["Train Accuracy", "Test Accuracy", "Gap (train-test)", "CV Accuracy (mean)", "Overfitting (Y/N)"]].round(4)

A big gap between train and test accuracy signals overfitting. Unconstrained trees and tree ensembles typically reach ~100% on the training data
and therefore get flagged; the **practical question is the absolute test score and whether it matches CV accuracy**.

### Detailed evaluation of the best model so far

In [ ]:
best_name = results_df["F1 (macro)"].idxmax()
best_pipe = fitted[best_name]
y_pred = best_pipe.predict(X_test)
print("Best untuned model (by macro-F1):", best_name, "\n")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

The classification report gives **precision, recall and F1 for each class** plus macro and weighted averages.
Per-class numbers show where the model struggles, which a single accuracy figure would hide.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(1, 2, figsize=(20, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title(f"{best_name}: confusion matrix (counts)")
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, normalize="true"), display_labels=class_names).plot(
    ax=ax[1], cmap="Blues", colorbar=False, values_format=".2f")
ax[1].set_title(f"{best_name}: confusion matrix (row-normalised = recall)")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

The confusion matrix shows which classes get mixed up. The counts view shows absolute errors; the row-normalised view shows per-class recall.
In the reference run, the largest error pool is **Dermason <-> Sira**, followed by **Barbunya <-> Cali**, while **Bombay is never confused** with any other class. Sira has the lowest F1 of all classes.

# 8. Hyperparameter Tuning

We tune the **three best-performing model families** (by cross-validated accuracy) with `RandomizedSearchCV`
(5-fold stratified CV, optimising macro-F1). Randomized search explores a wide space far faster than an exhaustive grid.
The search runs on the training set only.

In [ ]:
param_spaces = {
    "SVM": {"clf__C": loguniform(1e-1, 1e3), "clf__gamma": loguniform(1e-3, 1e0)},
    "Random Forest": {"clf__n_estimators": randint(200, 600), "clf__max_depth": [None, 10, 20, 30],
                      "clf__min_samples_split": randint(2, 12), "clf__min_samples_leaf": randint(1, 5),
                      "clf__max_features": ["sqrt", "log2", 0.5]},
    "Extra Trees": {"clf__n_estimators": randint(200, 600), "clf__max_depth": [None, 10, 20, 30],
                    "clf__min_samples_split": randint(2, 12), "clf__max_features": ["sqrt", "log2", 0.5]},
    "Hist Gradient Boosting": {"clf__learning_rate": loguniform(0.02, 0.3), "clf__max_iter": randint(100, 400),
                               "clf__max_depth": [None, 4, 6, 8], "clf__l2_regularization": loguniform(1e-3, 10),
                               "clf__min_samples_leaf": randint(10, 60)},
    "Logistic Regression": {"clf__C": loguniform(1e-2, 1e2)},
    "KNN": {"clf__n_neighbors": randint(3, 25), "clf__weights": ["uniform", "distance"], "clf__p": [1, 2]},
}

# top 3 model families (by CV accuracy) that have a search space defined
to_tune = [m for m in results_df.sort_values("CV Accuracy (mean)", ascending=False).index if m in param_spaces][:3]
print("Models to tune:", to_tune)

searches = {}
for name in to_tune:
    t0 = time.time()
    rs = RandomizedSearchCV(make_pipeline(get_models()[name]), param_spaces[name], n_iter=20, cv=skf,
                            scoring="f1_macro", n_jobs=-1, random_state=RANDOM_STATE, refit=True)
    rs.fit(X_train, y_train)
    searches[name] = rs
    print(f"{name:<24} best CV macro-F1 = {rs.best_score_:.4f}   ({time.time() - t0:.0f}s)")

For each shortlisted model, 20 random hyperparameter combinations are evaluated with 5-fold CV. `refit=True` retrains the best configuration on the full training set.

### Best parameters and improvement

In [ ]:
tuned_rows = []
for name, rs in searches.items():
    row, _ = evaluate(f"{name} (tuned)", rs.best_estimator_, cv=False)
    before = results_df.loc[name]
    row["CV F1 macro (tuned)"] = rs.best_score_
    row["Test Acc before"] = before["Test Accuracy"]
    row["Test Acc change"] = row["Test Accuracy"] - before["Test Accuracy"]
    row["Macro-F1 before"] = before["F1 (macro)"]
    row["Macro-F1 change"] = row["F1 (macro)"] - before["F1 (macro)"]
    tuned_rows.append(row)
    print(f"\n{name}\n  best params: { {k.replace('clf__', ''): (round(float(v), 5) if isinstance(v, (float, np.floating)) else v) for k, v in rs.best_params_.items()} }")

tuned_df = pd.DataFrame(tuned_rows).set_index("Model")
tuned_df[["Test Acc before", "Test Accuracy", "Test Acc change", "Macro-F1 before", "F1 (macro)",
          "Macro-F1 change", "Train Accuracy", "Gap (train-test)", "Overfitting (Y/N)"]].round(4)

The best parameters for each tuned model are printed, and the table compares test accuracy and macro-F1 **before vs after tuning**.
Small positive changes are expected because the default models are already strong on this dataset.

### Choosing the final model

In [ ]:
# Select with cross-validation (not the test set) so the test set stays an unbiased final check
final_name = max(searches, key=lambda n: searches[n].best_score_)
final_pipe = searches[final_name].best_estimator_
final_pred = final_pipe.predict(X_test)
print(f"Final model: {final_name} (tuned) | best CV macro-F1 = {searches[final_name].best_score_:.4f}")
print(f"Test accuracy = {accuracy_score(y_test, final_pred):.4f} | Test macro-F1 = {f1_score(y_test, final_pred, average='macro'):.4f}\n")
print(classification_report(y_test, final_pred, target_names=class_names, digits=4))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y_test, final_pred, normalize="true"), display_labels=class_names).plot(
    ax=ax, cmap="Greens", colorbar=False, values_format=".2f")
ax.set_title(f"Final model ({final_name}, tuned): normalised confusion matrix"); plt.xticks(rotation=30); plt.show()

The final model is the tuned candidate with the best **cross-validated** macro-F1. Choosing by CV rather than by test score keeps the held-out test set as an honest final estimate of real-world performance.

# 9. Model Comparison Table

In [ ]:
def to_table(df_):
    out = df_.reset_index()[["Model", "Train Accuracy", "Test Accuracy", "F1 (weighted)", "Overfitting (Y/N)"]]
    return out.rename(columns={"F1 (weighted)": "F1 Score"})

comparison = pd.concat([to_table(results_df), to_table(tuned_df)], ignore_index=True)
comparison = comparison.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

final_row = pd.DataFrame([{
    "Model": f"BEST MODEL -> {final_name} (tuned)",
    "Train Accuracy": accuracy_score(y_train, final_pipe.predict(X_train)),
    "Test Accuracy": accuracy_score(y_test, final_pred),
    "F1 Score": f1_score(y_test, final_pred, average="weighted"),
    "Overfitting (Y/N)": "Y" if accuracy_score(y_train, final_pipe.predict(X_train)) - accuracy_score(y_test, final_pred) > OVERFIT_GAP else "N",
}])
comparison = pd.concat([comparison, final_row], ignore_index=True)
comparison.to_csv("model_comparison.csv", index=False)
comparison.round(4)

The final table follows the project template: **Model, Train Accuracy, Test Accuracy, F1 Score (weighted), Overfitting (Y/N)**, listing every baseline and tuned model and ending with the **Best Model** row.
Overfitting is flagged when the train-test accuracy gap exceeds 3 percentage points. The table is also saved to `model_comparison.csv`.

# 10. Build a Simple Classifier App (Streamlit)

The final pipeline (capping + skew treatment + scaling + classifier) is saved together with the label encoder and a few reference values,
so the app only needs raw bean measurements as input.

In [ ]:
final_for_app = clone(final_pipe)
if "probability" in final_for_app.named_steps["clf"].get_params():       # SVM: enable class probabilities
    final_for_app.set_params(clf__probability=True)
final_for_app.fit(X, y)       # refit the chosen configuration on ALL labelled data for deployment

bundle = {
    "pipeline": final_for_app,
    "label_encoder": le,
    "features": features,
    "feature_stats": X.describe().T[["min", "50%", "max"]],
    "class_examples": df.groupby("Class")[features].median(),      # a typical bean per class
}
joblib.dump(bundle, "bean_classifier.joblib")
print("Saved bean_classifier.joblib")

# Same bundle as a plain .pkl file (standard-library pickle)
import pickle
with open("bean_classifier.pkl", "wb") as f:
    pickle.dump(bundle, f)
print("Saved bean_classifier.pkl")

The deployment model is refit on all labelled data using the hyperparameters chosen above (test scores in this notebook come from the train-only model).
For SVM, `probability=True` is switched on so the app can show class probabilities.

In [ ]:
APP_CODE = r'''
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Dry Bean Classifier", page_icon="🫘", layout="wide")


@st.cache_resource
def load_bundle():
    return joblib.load("bean_classifier.joblib")


bundle = load_bundle()
model, le = bundle["pipeline"], bundle["label_encoder"]
features, fstats, examples = bundle["features"], bundle["feature_stats"], bundle["class_examples"]

st.title("🫘 Dry Bean Type Classifier")
st.write("Enter the physical measurements of a bean (as produced by the camera system) and get the predicted bean type.")

st.sidebar.header("Quick start")
choice = st.sidebar.selectbox("Pre-fill with a typical bean of type:", ["Manual entry"] + list(examples.index))
st.sidebar.caption("Values are class medians from the training data. Edit any value afterwards.")

defaults = fstats["50%"] if choice == "Manual entry" else examples.loc[choice]

cols = st.columns(4)
values = {}
for i, f in enumerate(features):
    lo, hi = float(fstats.loc[f, "min"]), float(fstats.loc[f, "max"])
    step = max((hi - lo) / 500, 1e-6)
    values[f] = cols[i % 4].number_input(
        f, min_value=0.0, value=float(defaults[f]), step=step,
        format="%.4f" if hi <= 10 else "%.2f", key=f"{choice}_{f}",
        help=f"Training range: {lo:.4g} to {hi:.4g}")

if st.button("Predict bean type", type="primary"):
    X_new = pd.DataFrame([values], columns=features)
    label = le.inverse_transform(model.predict(X_new))[0]
    st.success(f"Predicted bean type: **{label}**")
    if hasattr(model, "predict_proba"):
        proba = pd.Series(model.predict_proba(X_new)[0], index=le.classes_, name="Probability")
        st.bar_chart(proba)
        st.caption(f"Confidence: {proba.max():.1%}")
'''
Path("app.py").write_text(APP_CODE.strip() + "\n", encoding="utf-8")
print("Wrote app.py")

`app.py` is a small Streamlit interface: 16 numeric inputs (pre-filled with the dataset median or a typical bean of any class), a **Predict** button,
the predicted bean type and, when the model supports it, a probability bar chart.

**To run the app** (in a terminal, from the folder containing `app.py`, `bean_classifier.joblib` and `bean_utils.py`):
```bash
pip install streamlit
streamlit run app.py
```

In [ ]:
# Sanity check: reload the saved bundle exactly as the app does and predict a "typical Bombay bean"
loaded = joblib.load("bean_classifier.joblib")
sample = loaded["class_examples"].loc[["Bombay"]]
print("Predicted:", loaded["label_encoder"].inverse_transform(loaded["pipeline"].predict(sample))[0])

The saved model is reloaded from disk and used on a typical Bombay bean, confirming the app will work when it loads the same file.

# Conclusion

**Results (reference run, 80/20 stratified split, seed 42)**
- 13,611 raw rows -> 13,543 after removing 68 duplicates; no missing values.
- Most classifiers land between **91% and 92.5% test accuracy**. Only AdaBoost (82.5%) and the simple Decision Tree (89.7%) / LDA (89.5%) / Naive Bayes (90.0%) lag behind.
- **Best model: tuned SVM (RBF kernel)** with about **92.4% test accuracy and 93.5% macro-F1**, no meaningful overfitting (train 93.9% vs test 92.4%), and the best cross-validated macro-F1 of the tuned candidates (about 94.4%).
- Tuned Logistic Regression matches it on test accuracy (92.4%), which shows the classes are nearly separable by smooth boundaries once features are scaled and de-skewed. Tree ensembles reach ~100% train accuracy and are flagged as overfitting without gaining test accuracy.
- Hyperparameter tuning gave only small gains (at most about +0.5 point), which is within the noise of a single 2,709-row test set.
- **Bombay is classified perfectly**; the remaining errors are mostly **Dermason <-> Sira** and **Barbunya <-> Cali**.
- Class imbalance is mild and did not need correction for this data.

**Business takeaway:** a ~92% accurate automated classifier based on 16 camera features can replace most manual sorting; flagging low-confidence predictions (e.g. probability < 70%) for manual checks would catch many of the Dermason/Sira confusions.

**Ideas to improve further:** engineer ratio features that separate Dermason from Sira, add image-based features (colour/texture), try stacking, or collect more data for the confusable classes.